In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r'D:\Desktop\test_score\data\train.csv').drop('id', axis=1)

EDA

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('exam_score', axis=1)
y = df['exam_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42
)

In [ ]:
numeric  = X.select_dtypes(exclude='object').columns.tolist()
category = X.select_dtypes(include='object').columns.tolist()

Encoding Categories

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ORDINAL ENCODING
ord_features = ['sleep_quality', 'facility_rating', 'exam_difficulty']
ord_pipeline = Pipeline([
    ('ord_pipeline', OrdinalEncoder(categories=[
        ['poor', 'average', 'good'],
        ['low', 'medium', 'high'],
        ['easy', 'moderate', 'hard']
    ]))
])

# ONEHOT ENCODING
ohe_features = ['internet_access']
ohe_pipeline = Pipeline([
    ('ohe_pipeline', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# TARGET ENCODING
target_features = ['gender', 'study_method']
target_pipeline = Pipeline([
    ('target_pipeline', TargetEncoder(cv=5, smooth=7, random_state=42))
])

prep = ColumnTransformer([
    ('ord', ord_pipeline, ord_features),
    ('ohe', ohe_pipeline, ohe_features),
    ('target', target_pipeline, target_features),
    ('num', 'passthrough', numeric)
],
# remainder='passthrough',
verbose_feature_names_out=False
).set_output(transform='pandas')

X_train = prep.fit_transform(X_train, y_train)
X_test  = prep.transform(X_test)

In [ ]:
# просмотрим частное содержание признаков
for col in numeric + category:
    print(X[col].value_counts(normalize=True) * 100)
    print('\n')

FEATURES ENGINEERING

In [ ]:
import numpy as np

# XGB любит логические признаки (создаем масками)

def features_eng(all_data):

    for data in all_data:
        
        data['study_hours_clip'] = data['study_hours'].clip(0, 10)

        # data['study_hours_low']  = np.maximum(0, 5 - data['study_hours'])
        # data['study_hours_mid']  = np.maximum(0, data['study_hours'] - 5)
        # data['study_hours_high'] = np.maximum(0, data['study_hours'] - 10)


        data['sleep_hours_clip'] = data['sleep_hours'].clip(4, 9)

        # data['sleep_hours_low'] = np.maximum(0, 4 - data['sleep_hours'])
        # data['sleep_hours_mid'] = np.maximum(0, data['sleep_hours'] - 4)
        # data['sleep_hours_high'] = np.maximum(0, data['study_hours'] - 9)

        # data['class_attendance_clip'] = data['class_attendance'].clip(5, 95)


        # data['study_hours_sleep_hours'] = data['study_hours_clip'] / (data['sleep_hours_clip'] + 1e-5)
        data['internet_study'] = data['internet_access_yes'] * data['study_hours_clip']
        data['facility_attendance'] = data['facility_rating'] * data['class_attendance']


        # data['course_class_attendance'] = data['course'] * data['class_attendance']

        # for clock in [1,2,3,4,5,6,7]:
            # data[f'sleep_hours_deficit_{clock}'] = np.maximum(0, clock - data['sleep_hours_clip'])

        # for clock in [3,6,7]:
            # data[f'study_deficit_{clock}'] = np.maximum(0, clock - data['study_hours_clip'])

        data['study_deficit_7'] = np.maximum(0, 7 - data['study_hours_clip'])

        # for t in [4, 4.5, 5]:
        #     data[f'study_deficit_{t}'] = np.maximum(0, t - data['study_hours_clip'])

        data.drop(
            [
                 'study_hours',
                 'sleep_hours',
                 'internet_access_yes',
                 'facility_rating',
                 'age',
                 'gender',
                 'exam_difficulty',
            ],
            axis=1, inplace=True
        )

        # # хоть он и в ordinal закодированном порядке, но мне кажется лучше показать сети, что сложность - находтся не просто на линейном расстоянии друг от друга
        # data['exam_difficulty'] = data['exam_difficulty'] ** 2
        # # аналогично с качеством сна
        # data['sleep_quality'] = data['sleep_quality'] ** 2
        # data['facility_rating'] = data['facility_rating'] ** 2

        # data['exam_study'] = data['exam_difficulty'] * data['study_hours']
        # data['exam_sleep'] = data['exam_difficulty'] * data['sleep_hours']
        # data['sleep_age'] = data['sleep_hours'] * data['age']

In [ ]:
features_eng( (X_train, X_test) )

SEARCHING HYPERPARAMS

In [ ]:
# from xgboost import XGBRegressor

# import optuna

# from sklearn.metrics import root_mean_squared_error as rmse


# def objective(trial):

#     params1 = {
#         'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
#         # 'n_estimators': 2000,
#         'learning_rate': trial.suggest_float('learning_rate', .005, .3, log=True),
#         'max_depth': trial.suggest_int('max_depth', 3, 12)
#     }
#     params2 = {
#         'subsample': trial.suggest_float('subsample', .4, 1.),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', .4, 1.),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 20)
#     }
#     params3 = {
#         'reg_alpha': trial.suggest_float('reg_alpha', 0., 10.),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1., 10.)
#     }
#     # # params4 = {
#     # #     'gamma': trial.suggest_int('gamma', 0., 5.)
#     # #     # 'scale_pos_weight': trial.suggest_int
#     # # }


#     # будем пробовать разные параметры
#     model = XGBRegressor(
#         **params1, **params2, **params3,
#         early_stopping_rounds=50,
#         objective='reg:squarederror',
#         eval_metric='rmse',
#         n_jobs=4, random_state=42
#     )

#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_valid, y_valid)],
#         verbose=False,
#         # callbacks=[optuna.integration.XGBoostPruningCallback(trial, 'validation_0-rmse')]
#     )

#     scores = model.evals_result()['validation_0']['rmse']

#     for epoch, score in enumerate(scores, 1):

#         trial.report(score, epoch)

#         if trial.should_prune():
#             raise optuna.TrialPruned()
        
#     return model.best_score


# from optuna.pruners import MedianPruner

# study = optuna.create_study(
#     direction='minimize',
#     # sampler=optuna.samplers.CmaEsSampler(),
#     sampler=optuna.samplers.TPESampler(),
#     pruner=MedianPruner(
#         n_startup_trials=5,
#         n_warmup_steps=10
#     )
# )

# params = {
#     'n_estimators': 679,
#     'learning_rate': 0.11604488937329954,
#     'max_depth': 6,
#     'subsample': 0.8823741336030094,
#     'colsample_bytree': 0.6694833904422619,
#     'min_child_weight': 8,
#     'reg_alpha': 4.887201433121097,
#     'reg_lambda': 4.500445906986548
# }

# study.enqueue_trial(params)

# study.optimize(
#     objective,
#     n_trials=30, timeout=30*60,
#     n_jobs=4,
#     gc_after_trial=True
# )

In [ ]:
# study.best_params

In [ ]:
# study.best_value

In [ ]:
from xgboost import XGBRegressor

params = {
    'n_estimators': 679,
    'learning_rate': 0.11604488937329954,
    'max_depth': 6,
    'subsample': 0.8823741336030094,
    'colsample_bytree': 0.6694833904422619,
    'min_child_weight': 8,
    'reg_alpha': 4.887201433121097,
    'reg_lambda': 4.500445906986548,
    'gamma': 3
}

model = XGBRegressor(
    **params,
    n_jobs=4, random_state=42
)

LOCAL TESTING (without)

In [ ]:
X_complete = X_train
y_complete = y_train

# model.fit(X_complete, y_complete)

# from sklearn.metrics import root_mean_squared_error as rmse

# print(rmse(y_test, model.predict(X_test)))

KAGGLE KFOLD TESTIING

In [ ]:
from sklearn.metrics import root_mean_squared_error as rmse

X_complete = pd.concat([X_complete, X_test])
y_complete = pd.concat([y_complete, y_test])

df_ = pd.read_csv(r'D:\Desktop\test_score\data\test.csv')
test_df = df_.copy()
test_idx = test_df['id']
test_df = test_df.drop('id', axis=1)

test_df = prep.transform(test_df)

features_eng( (test_df,) )


from sklearn.model_selection import KFold

kf = KFold(n_splits=30, shuffle=True, random_state=42)

X_complete_preds = np.zeros(len(X_complete))
xgb_test_preds   = np.zeros(len(test_df))

for i, (train_idx, valid_idx) in enumerate(kf.split(X_complete), 1):

    X_tr, y_tr   = X_complete.iloc[train_idx], y_complete.iloc[train_idx]
    X_val, y_val = X_complete.iloc[valid_idx], y_complete.iloc[valid_idx]

    model.fit(X_tr, y_tr)

    # заполняем по очереди каждый валидационный Fold (новый столбец признаков)
    # X_complete_preds[valid_idx] = model.predict(X_val)

    # *необязательно, но посмотрим метрику на одном fold-е
    print(f'Метрика на {i}\'s fold - {rmse(y_val, model.predict(X_val)):.5f}')

    # делаем предсказания для вычисления средней метрики xgb на данных для submissions
    xgb_test_preds += model.predict(test_df) / kf.get_n_splits()

# # тестируем уже на отдельном тренировочном. Только xgb
# print(f'Тестовая xgb - {rmse(y_test, model.predict(X_test))}')

# # добавим в тренировочные данные столбец предсказаний (получается мета данные для тренировки)
# X_complete['xgb_preds'] = X_complete_preds

# # добавим в submit данные новый признак
# test_df['xgb_preds'] = xgb_test_preds

SUBMISSION FOR KAGGLE

In [ ]:
submit = pd.DataFrame({
    'id': test_idx,
    'exam_score': xgb_test_preds
    }
).to_csv('submission_31.csv', index=False)

In [ ]:
importances = pd.Series(
    index = model.feature_names_in_,
    data = model.feature_importances_
).sort_values(ascending=False)
importances

**ATTEMPT STACKING MODELS (XGBRegressor, Ridge)

In [ ]:
# from sklearn.preprocessing import StandardScaler

# # X_complete = X_complete.drop(['gender', 'study_method'], axis=1)
# # test_df = test_df.drop(['gender', 'study_method'], axis=1)

# # X_complete['gender'] = df['gender'].values
# # X_complete['study_method'] = df['study_method'].values

# # test_df['gender'] = df_['gender'].values
# # test_df['study_method'] = df_['study_method'].values


# # ohe_pipeline = Pipeline([
# #     ('ohe_pipeline', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
# # ])

# # # без бинарных признаков
# # numeric = X_complete.select_dtypes(exclude='object').columns.tolist()


# prep2 = ColumnTransformer([
#     # ('ohe', ohe_pipeline, ['gender', 'study_method']),
#     ('num', StandardScaler(), ['xgb_preds'])
# ],
# remainder='drop').set_output(transform='pandas')


# X_complete = prep2.fit_transform(X_complete)
# test_df    = prep2.transform(test_df)

In [ ]:
# # сейчас нужно обучить на обновленных данных meta-модель
# from sklearn.linear_model import Ridge, RidgeCV, ElasticNetCV
# from sklearn.model_selection import cross_val_predict as cvs


# # для важности признаков
# features_name = X_complete.columns.tolist()

# # RIDGE MODEL
# meta_model = RidgeCV(
#     alphas=np.logspace(-3, 3, 50),
#     cv=5, scoring='neg_root_mean_squared_error',
#     # store_cv_values=True
# )
# # meta_model = Ridge(alpha=0.1)

# # ELASTIC NET
# # meta_model = ElasticNetCV(
# #     l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
# #     alphas=np.logspace(-4, 1, 50),
# #     cv=5
# # )

# meta_model.fit(X_complete, y_complete)

# # print(f'Тестовая ridge - {rmse(y_test, meta_model.predict(X_test))}')

# # а теперь, можно посмотреть и полный submit
# submission = pd.DataFrame({
#     'id': test_idx,
#     'exam_score': meta_model.predict(test_df)
# }).to_csv('submission_18.csv', index=False)

In [ ]:
# features_name

In [ ]:
# importances = pd.Series(
#     index = features_name,
#     data = meta_model.coef_
# ).sort_values(ascending=False)
# importances